<a href="https://colab.research.google.com/github/jesobczyk/NustecSchool2026/blob/main/L3_HartreeFock/L3_SkyrmeHF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lecture 3 — your own Hartree–Fock solver

In Lecture 2 the nucleus was a **Woods–Saxon well**: a guessed potential with a depth, a radius and a surface, fitted to (e,e′p) data. Here the well is **not guessed**. It is made by the nucleons themselves, from a nuclear force, by the Hartree–Fock loop of Lecture 2:

$$\rho(\mathbf r)=\sum_{\alpha\,\in\,\text{occ}}|\varphi_\alpha(\mathbf r)|^2
\;\longrightarrow\; U[\rho](\mathbf r)
\;\longrightarrow\; \Big[-\frac{\hbar^2}{2m}\nabla^2+U(\mathbf r)+\text{spin–orbit}\Big]\varphi_\alpha=\varepsilon_\alpha\varphi_\alpha
\;\longrightarrow\;\text{new }\rho \;\longrightarrow\;\dots$$

repeated until nothing changes. The solver in `hf_core.py` does exactly this, for spherical nuclei, in about a second.

**What you will see**

- **A**: the loop converging, and what happens when you remove the piece of the force that gives saturation.
- **B**: the density and the mean field it makes.
- **C**: the orbitals, the magic numbers with and without spin–orbit, and a comparison with (e,e′p).
- **D**: the orbitals in momentum space, against a local Fermi gas with the same density, and the spectral-function stripes of Lecture 2.
- **E**: a nucleon leaving through the same field, against a plane wave.

Run the cell below once, then each panel. If the sliders do not appear, set `L.USE_WIDGETS = False` and every panel draws static figures instead.

In [1]:
#@title ▶ Run this cell first (sets up the notebook on Google Colab; does nothing on your own computer)
import os, sys
if 'google.colab' in sys.modules:
    if not os.path.isdir('/content/NustecSchool2026'):
        !git clone -q https://github.com/jesobczyk/NustecSchool2026.git /content/NustecSchool2026
    %cd /content/NustecSchool2026/L3_HartreeFock
    from google.colab import output
    output.enable_custom_widget_manager()
%matplotlib inline

### Running on your own computer (skip this on Google Colab)

The notebook needs `numpy`, `scipy`, `matplotlib` and `ipywidgets` (the package that draws the sliders). Install them once, in a terminal:

```bash
pip install numpy scipy matplotlib ipywidgets jupyterlab
```

or, if you use conda:

```bash
conda install -c conda-forge numpy scipy matplotlib ipywidgets jupyterlab
```

From the repository folder, `pip install -r requirements.txt` does the same. You can also install from inside the notebook by running `%pip install ipywidgets` in a cell, and then restarting the kernel.

Then start Jupyter with `jupyter lab` and open this notebook.

- **JupyterLab 3 or newer, Notebook 7 or newer, and VS Code** show the sliders as soon as `ipywidgets` is installed in the same Python environment as the kernel.
- **Classic Jupyter Notebook (older than version 7):** also run `jupyter nbextension enable --py widgetsnbextension`, then restart Jupyter.
- If you see *"Error displaying widget"* or no sliders: restart the kernel (and the Jupyter server). As a last resort, uncomment `L.USE_WIDGETS = False` in the next cell to get static versions of all panels.

In [2]:
import hf_l3 as L       # panels;  the physics is in hf_core.py
import hf_core as H
# L.USE_WIDGETS = False  # uncomment if the sliders do not appear

## The force: Skyrme in five terms

A realistic nuclear force (e.g. from chiral EFT) is too expensive for a one-second calculation. The **Skyrme force** is an effective force with a few terms. Each one plays a role:

| term | what it is | what it does |
|---|---|---|
| $t_0$ | zero-range attraction | binds the nucleus |
| $t_3\,\rho^{\alpha}$ | repulsion that grows with density | **saturation**: stops the collapse (it stands in for the three-nucleon force) |
| $t_1,\,t_2$ | momentum dependence | the surface |
| $W_0$ | spin–orbit | splits $j=\ell\pm\tfrac12$: the **magic numbers** of Lecture 2 |
| Coulomb | protons only | pushes the proton levels up |

Because the force has zero range, the energy is an integral of a function of local density.

$$E=\int d^3r\;\mathcal H\big(\rho(\mathbf r)\big),$$

and the mean field is its derivative, $U_q=\delta E/\delta\rho_q$. Two parameter sets are included: **SLy4** (Chabanat et al. 1998) and **SkM\*** (Bartel et al. 1982). Both were fitted to binding energies and radii of a few doubly magic nuclei and to nuclear matter — not to the nucleus you are computing.

## A — The loop

The calculation starts from a Woods–Saxon guess, builds the density, the fields, new orbitals, and repeats. Each iteration mixes 40 % of the new density into the old one, which keeps the loop stable.

Left: the total energy, with the measured binding energy. Middle: the size of the nucleus. Right: the energies of the occupied orbitals.

Then move the $t_3$ slider down: you remove part of the repulsion that grows with density.

**Questions**

1. How many iterations does ¹⁶O need? Which quantity settles first?
2. Set $t_3 \times 0.9$, then $0.75$, then $0$. What happens to the energy and the radius? What does this say about saturation — the constant density $\rho_0$ of Lecture 2?

In [ ]:
L.show_A()

## B — The density and the field it makes

Left: neutron, proton and total densities. The interior density is close to $\rho_0 = 0.16$ fm⁻³.

Middle: the mean field $U(r)$ for neutrons and protons, against the Woods–Saxon guess the loop started from.


**Questions**

3. What is the central depth of the field for ⁴⁰Ca? How does it compare with the 50–60 MeV of the Woods–Saxon wells of Lecture 2?
4. Why is the proton field less deep than the neutron field, and why does it not go to zero outside?

In [ ]:
L.show_B()

## C — Orbitals and magic numbers

Each spherical orbital is labelled $(n,\ell,j)$ and holds $2j+1$ nucleons (Lecture 2). Thick lines are occupied, dashed lines are empty. For ¹²C, the protons are compared with the (e,e′p) removal energies of Mougey et al. (1976).

**Questions**

5. ⁴⁰Ca: set $W_0 \times 0$. Which levels become degenerate? Which shell gaps survive, and which disappear?
6. ¹²C: compare the proton 1p3/2 and 1s1/2 with (e,e′p). Which is closer? 

In [5]:
L.show_C()

Output()

## D — Momentum space and the spectral function

The momentum distribution of each orbital is the Fourier (Bessel) transform of its radial function (Lecture 2):

$$\tilde\varphi_{n\ell j}(p)=\sqrt{\tfrac{2}{\pi}}\int_0^\infty dr\;r\,u_{n\ell j}(r)\,j_\ell(pr/\hbar c),\qquad
n(p)=\frac{1}{4\pi}\sum_{\text{occ}}(2j+1)\,|\tilde\varphi_{n\ell j}(p)|^2 .$$

Middle: the total against a **local Fermi gas built from the same density**, $n(k)=\frac{1}{4\pi^3}\int d^3r\,\theta\big(k_F(r)-k\big)$ with $k_F(r)=(3\pi^2\rho_q(r))^{1/3}$. Right: the spectral function of the orbitals — each orbital is a stripe at its removal energy $-\varepsilon_\alpha$, smeared by a width $\Gamma$.

**Questions**

7. Where does the HF momentum distribution have strength that the local Fermi gas does not? Is there anything above $\approx 300$ MeV in either?
8. What spectroscopic factors does a Hartree–Fock calculation produce?

In [4]:
L.show_D()

Output()

## BONUS E — A nucleon leaving through the same field

In (e,e′p) and in neutrino scattering the struck nucleon has to leave the nucleus. The simplest consistent picture: it moves through **the same mean field** that bound it. The panel solves the radial equation at positive energy $T$ in the HF field and compares it with a plane wave, both normalised to the same amplitude far away. The difference far away is a phase shift; inside the nucleus the wave is also distorted.

**Questions**

9. For ⁴⁰Ca and ℓ = 0, compare $T$ = 50, 150 and 300 MeV. When is the plane wave a good approximation inside the nucleus?
10. This field is real. What is missing compared with the optical potential and the transparency?

In [5]:
L.show_E()

Output()

## Answers

Numbers from the solver with SLy4 unless stated otherwise.

### A — The loop

**1.** About 30 iterations (to 10⁻⁶ MeV). The orbital energies settle first, within about 5 iterations; the energy and the radius are within 0.1 MeV and 0.01 fm after about 8.

**2.**

| t₃ × | E [MeV] | r_ch [fm] |
|---|---|---|
| 1 | −128.6 | 2.79 |
| 0.9 | −235 | 2.45 |
| 0.75 | −518 | 2.08 |
| 0.6 and below | no stable solution: the nucleus keeps shrinking and the energy keeps falling | |

Without enough repulsion that grows with density, nothing stops the nucleus from contracting. Saturation — the same interior density ρ₀ and binding per nucleon in every nucleus — needs this term; in a realistic Hamiltonian the three-nucleon force plays that role.

### B — The density and the field it makes

**3.** About −60 MeV at the centre for neutrons (minimum ≈ −67 MeV just off centre); −52 MeV with SkM\*. The same range as the Woods–Saxon wells of Lecture 2, but now an output of the force, not an input.

**4.** Coulomb: the protons repel each other, which adds about +10 MeV inside. Outside, the Coulomb potential falls only as 1/r, so the proton field stays positive (≈ +2 MeV at the edge of the grid) instead of going to zero.


### C — Orbitals and magic numbers

**5.** Each pair j = ℓ ± ½ becomes degenerate (1p at −33.7 MeV, 1d at −19.3 MeV, 1f at −5.9 MeV). The gaps at 2, 8 and 20 survive: they do not need spin–orbit. The next closure is then 40 (after 1f and 2p). With spin–orbit the 1f7/2 separates from the 2p3/2 and 28 appears.

**6.**

| orbital | SLy4 | SkM\* | (e,e′p) |
|---|---|---|---|
| 1p3/2 | −14.1 MeV | −14.1 MeV | 17.5 MeV |
| 1s1/2 | −31.2 MeV | −29.0 MeV | 38.1 MeV |


### D — Momentum space and the spectral function

**7.** The orbitals have strength above k_F(0) ≈ 260 MeV, which the local Fermi gas cannot have; it comes from confining the nucleons to a finite region (localisation). Both are essentially zero above ≈ 350–400 MeV.

**8.** A single Slater determinant has occupations exactly 0 or 1: every orbital below the Fermi level is completely full. No Hartree–Fock calculation can therefore deplete the orbitals or put strength at high momentum and high removal energy. That needs correlations beyond a single determinant, from the short-range repulsion and the tensor force: the next part of the lecture.

### E — A nucleon leaving through the same field

**9.** For ⁴⁰Ca and ℓ = 0 the largest difference from the plane wave inside the nucleus is about 1.7 at 50 MeV (phase shift ≈ 111°), about 0.2 at 150 MeV (≈ 6°) and about 1.0 at 300 MeV (≈ −67°). At 150 MeV the plane wave is nearly right inside. At 300 MeV the distortion returns.

**10.** The Hartree–Fock field is real, so no flux is lost: the transparency would be 1. An optical potential has an imaginary part that describes the outgoing nucleon knocking out or exciting other nucleons — the absorption seen as T ≈ 0.6 in (e,e′p) (Lecture 2). Getting it requires coupling to other channels: a continuum RPA calculation, an optical potential, or a cascade (Lecture 4).